# Activity 3: ResNet-18 with a Poincaré BMLR Head

[![QR code for the MLSS RDL tutorial repository](https://raw.githubusercontent.com/GitZH-Chen/MLSS-RDL-Tutorial/main/assets/github-repository-qr-260.png)](https://github.com/GitZH-Chen/MLSS-RDL-Tutorial)

**GitHub repository:** [github.com/GitZH-Chen/MLSS-RDL-Tutorial](https://github.com/GitZH-Chen/MLSS-RDL-Tutorial)

Many applications use a standard Euclidean encoder to extract features, then map those features to hyperbolic space for downstream learning. 

Here we study a simple image-classification example on MNIST: ResNet-18 is the Euclidean backbone, and BMLR is the hyperbolic classification head:

$$
\text{image}\xrightarrow{\text{ResNet-18}}z\in\mathbb{R}^{512}
\xrightarrow{\operatorname{CLIP}+\operatorname{Exp}_0^K}x\in\mathbb{P}^{512}_K
\xrightarrow{\text{BMLR}}\text{class logits}.
$$



## Contents

1. [Mathematical recap](#scrollTo=bmlr-and-clipping-recap)
2. [Setup](#scrollTo=activity-3-setup)
3. [Load MNIST](#scrollTo=activity-3-data)
4. [Define ResNet-18 + Poincaré BMLR](#scrollTo=activity-3-model)
5. [Train, validate, and test](#scrollTo=activity-3-training)
6. [Modify and interpret the clipping radius](#scrollTo=activity-3-modify)
7. [Takeaways](#scrollTo=activity-3-takeaways)


## Mathematical recap

For class $k$, BMLR uses the logit introduced in Activity 2,

$$
u_k(x)=-\alpha_k B^{v_k}(x)+b_k,
\qquad
p(y=k\mid x)=\frac{\exp(u_k(x))}{\sum_j\exp(u_j(x))}.
$$

On the Poincaré ball with curvature $K<0$,

$$
B^v(x)=\frac{1}{\sqrt{-K}}\log\left(
\frac{\lVert v-\sqrt{-K}x\rVert^2}
{1+K\lVert x\rVert^2}
\right).
$$

ResNet produces a Euclidean feature $z$. We clip its norm and map it to the Poincaré ball in one step:

$$
x=\operatorname{Exp}_0^K\!\left(\min\left\{1,\frac{r}{\lVert z\rVert}\right\}z\right).
$$

Without clipping, large Euclidean features are mapped close to the boundary of the Poincaré ball, where gradients through the exponential map vanish and optimization becomes unstable. Clipping keeps the mapped features in a smaller interior region [Guo et al., 2022]. We use $K=-1$ and $r=1$.


**References**

- Ziheng Chen, Bernhard Schölkopf, and Nicu Sebe. Hyperbolic Busemann Neural Networks. CVPR 2026. [Paper](https://arxiv.org/abs/2602.18858) · [Code](https://github.com/GitZH-Chen/HBNN)
- Yunhui Guo, Xudong Wang, Yubei Chen, and Stella X. Yu. Clipped Hyperbolic Classifiers Are Super-Hyperbolic Classifiers. CVPR 2022. [Paper](https://openaccess.thecvf.com/content/CVPR2022/html/Guo_Clipped_Hyperbolic_Classifiers_Are_Super-Hyperbolic_Classifiers_CVPR_2022_paper.html)


## Setup


In [ ]:
%pip -q install geoopt==0.5.1

!test -d /content/mlss_hbnn || git clone -q https://github.com/GitZH-Chen/HBNN.git /content/mlss_hbnn
!git -C /content/mlss_hbnn checkout -q d5c79c8eed36a0b7c2f15e9a8fbcd0318216e5b0


In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

warnings.filterwarnings("ignore", category=SyntaxWarning)
sys.path.insert(0, "/content/mlss_hbnn")
from lib.bnn.BMLR import BMLR
from lib.bnn.Geometry import Stereographic

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## Load MNIST

We resize each grayscale image to $32\times32$, use 10,000 examples for training and 2,000 for validation, and evaluate the final model on the complete 10,000-image MNIST test set.


In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_data = datasets.MNIST("/content/data", train=True, download=True, transform=transform)
test_data = datasets.MNIST("/content/data", train=False, download=True, transform=transform)
train_loader = DataLoader(Subset(train_data, range(10_000)), batch_size=128, shuffle=True)
validation_loader = DataLoader(Subset(train_data, range(10_000, 12_000)), batch_size=256)
test_loader = DataLoader(test_data, batch_size=256)


## Define ResNet-18 + Poincaré BMLR

The only geometric hyperparameter exposed here is the clipping radius $r$.


In [ ]:
class ResNet18PoincareBMLR(nn.Module):
    def __init__(self, clip_radius=1.0, K=-1.0):
        super().__init__()
        self.encoder = models.resnet18(weights=None)
        self.encoder.conv1 = nn.Conv2d(1, 64, 3, stride=1, padding=1, bias=False)
        self.encoder.maxpool = nn.Identity()
        self.encoder.fc = nn.Identity()
        self.ball = Stereographic(K=K)
        self.head = BMLR(n_classes=10, dim=512, K=K, metric="poincare")
        self.clip_radius = clip_radius

    def forward(self, images):
        z = self.encoder(images)
        factor = torch.clamp(self.clip_radius / z.norm(dim=-1, keepdim=True), max=1.0)
        return self.head(self.ball.exp0(factor * z))


model = ResNet18PoincareBMLR(clip_radius=1.0, K=-1.0).to(device)


## Train, validate, and test

We train for 20 epochs and track the average training loss and validation accuracy after every epoch.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
loss_history = []
validation_accuracy_history = []

for epoch in range(1, 21):
    model.train()
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)

    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in validation_loader:
            predictions = model(images.to(device)).argmax(dim=1).cpu()
            correct += (predictions == labels).sum().item()

    train_loss = total_loss / len(train_loader.dataset)
    validation_accuracy = correct / len(validation_loader.dataset)
    loss_history.append(train_loss)
    validation_accuracy_history.append(validation_accuracy)
    print(
        f"Epoch {epoch:2d} | loss {train_loss:.4f} | "
        f"validation accuracy {validation_accuracy:.2%}"
    )


### Plot the learning curves


In [ ]:
epochs = range(1, 21)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(epochs, loss_history)
axes[0].set(xlabel="Epoch", ylabel="Cross-entropy", title="Training loss")
axes[1].plot(epochs, validation_accuracy_history)
axes[1].set(xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1.02), title="Validation accuracy")
for axis in axes:
    axis.set_xticks([1, 5, 10, 15, 20])
plt.tight_layout()
plt.show()


In [ ]:
model.eval()
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        predictions = model(images.to(device)).argmax(dim=1).cpu()
        correct += (predictions == labels).sum().item()

print(f"Test accuracy: {correct / len(test_loader.dataset):.2%}")


## Modify and interpret

Change `clip_radius=1.0` to `0.5` or `2.0`, recreate the model, and rerun training.

- A smaller $r$ clips more Euclidean features and keeps their Poincaré images closer to the origin.
- A larger $r$ allows features to move closer to the boundary, where the exponential map can produce smaller back-propagated gradients.

Interpret. The image encoder remains Euclidean. Only the classification head is hyperbolic: `CLIP → Exp₀ → BMLR`.


## Takeaways

- ResNet-18 maps MNIST images to 512-dimensional Euclidean features.
- Clipping controls the norm of each feature before the exponential map.
- The exponential map sends clipped features to the Poincaré ball.
- BMLR converts Poincaré features into ten class logits through Busemann functions.
- The complete hybrid model is trained end to end with ordinary cross-entropy loss.
